# 🫀 Heart Attack Risk Prediction - Exploratory Data Analysis & Machine Learning

This notebook provides a comprehensive end-to-end analytical workflow for the **Heart Attack Risk Prediction** dataset.

### Table of Contents
1. **Environment Setup & Data Ingestion**
2. **Exploratory Data Analysis (EDA) & Summary Statistics**
3. **Data Cleaning & Feature Engineering**
4. **Pipeline Construction & Preprocessing**
5. **Model Training & Comparison**
6. **Evaluation & Performance Metrics**
7. **Model Interpretability (Feature Importance & SHAP)**
8. **Key Clinical Insights & Conclusion**

## 1. Environment Setup & Data Ingestion

In [ ]:
import os
import sys
from pathlib import Path

# Setup project path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from preprocessing import load_data, clean_data, prepare_data

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Environment initialized successfully!')

In [ ]:
# Load raw dataset
df_raw = load_data()
print(f'Dataset Shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
df_raw.head()

## 2. Exploratory Data Analysis (EDA) & Summary Statistics

In [ ]:
# Dataset Info & Missing Value Check
print('--- Dataset Summary Info ---')
print(df_raw.info())
print('\n--- Missing Values Count ---')
print(df_raw.isnull().sum())

In [ ]:
# Target Variable Distribution
target_counts = df_raw['Heart Attack Risk'].value_counts()
target_pct = df_raw['Heart Attack Risk'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(x='Heart Attack Risk', data=df_raw, palette=['#2a9d8f', '#e76f51'], ax=ax[0])
ax[0].set_title('Target Class Counts', fontweight='bold')
ax[0].set_xticklabels(['No Risk (0)', 'High Risk (1)'])

ax[1].pie(target_counts, labels=['No Risk (0)', 'High Risk (1)'], autopct='%1.1f%%', 
         colors=['#2a9d8f', '#e76f51'], startangle=90, explode=(0, 0.05))
ax[1].set_title('Class Proportion', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Cleaning & Feature Engineering

In [ ]:
# Clean and prepare engineered features
df_clean = clean_data(df_raw)
print(f'Cleaned Data Shape: {df_clean.shape}')
df_clean[['Age', 'Systolic BP', 'Diastolic BP', 'Cholesterol', 'Triglycerides', 'BMI', 'Heart Attack Risk']].head()

In [ ]:
# Feature Correlation Heatmap
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(14, 10))
corr_matrix = df_clean[numeric_cols].corr()
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True, fmt='.2f', linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Clinical & Lifestyle Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 4. Pipeline Construction & Model Evaluation

In [ ]:
# Prepare Train/Test Split and ColumnTransformer
X_train, X_test, y_train, y_test, preprocessor = prepare_data(df_clean)
print(f'X_train Shape: {X_train.shape}')
print(f'X_test Shape:  {X_test.shape}')

In [ ]:
# Load evaluation comparison metrics from results/
metrics_path = PROJECT_ROOT / 'results' / 'metrics' / 'model_comparison.csv'
if metrics_path.exists():
    df_metrics = pd.read_csv(metrics_path)
    display(df_metrics)
else:
    print('Run evaluate.py to generate model comparison metrics.')

In [ ]:
# Visualizing Model Performance
if metrics_path.exists():
    plt.figure(figsize=(10, 5))
    sns.barplot(x='Model', y='ROC-AUC', data=df_metrics, palette='viridis')
    plt.title('Model Comparison: Test Set ROC-AUC Scores', fontsize=13, fontweight='bold')
    plt.ylim(0.5, 1.0)
    plt.ylabel('ROC-AUC Score')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

## 5. Model Explainability & Clinical Findings

In [ ]:
# Display saved Explainability Visualizations
from IPython.display import Image, display

fi_img = PROJECT_ROOT / 'results' / 'figures' / 'feature_importance.png'
shap_img = PROJECT_ROOT / 'results' / 'figures' / 'shap_summary.png'

if fi_img.exists():
    print('Top Feature Importance:')
    display(Image(filename=str(fi_img)))

if shap_img.exists():
    print('SHAP Summary Plot:')
    display(Image(filename=str(shap_img)))